In [15]:
from collections import Counter
import numpy as np
from sklearn import datasets
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

################################################################
##PLEASE IMPLEMENT THE FOLLOWING FUNCTIONS
##FUNCTION ARGUMANTS, DEFINITIONS FOR SOME OF THE FUNCTIONS, AND
## SOME HELP IS GIVEN.
#################################################################  


def accuracy(y_true, y_pred):
        accuracy = np.sum(y_true == y_pred) / len(y_true)
        return accuracy

def entropy(x):

    if len(x) == 0:
        return 0

    # Count the occurrences of each unique class in x
    class_counts = Counter(x)

    entropy_value = 0
    total_samples = len(x)

    for count in class_counts.values():
        probability = count / total_samples
        entropy_value -= probability * np.log2(probability)

    return entropy_value

    pass

class Node:
    def __init__(self, feature, threshold, left, right, *, value):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None

class TreeRegressor:
    def __init__(self, min_samples_split=2, max_depth=100, n_feats=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.root = None

    def fit(self, X, y):
        self.root = self.build_tree(X, y)

    def predict(self, X):

        predictions = []
        for x in X:
            prediction = self.traverse_tree(x, self.root)
            predictions.append(prediction)
        return np.array(predictions)
        pass

    def build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape

        # Stopping criteria:
        if depth >= self.max_depth or n_samples < self.min_samples_split:
            return Node(None,None,None,None,value=self.common_thing(y))

        # Try to find the best split
        best_split = self.get_best_split(X, y, feat_idxs=range(n_features))

        # If there's no best split, stop and return the common value
        if best_split is None:
            return Node(None,None,None,None,value=self.common_thing(y))

        feature, threshold, left_indices, right_indices, info_gain = best_split

        # Recursively build the left and right subtrees
        left_subtree = self.build_tree(X[left_indices], y[left_indices], depth + 1)
        right_subtree = self.build_tree(X[right_indices], y[right_indices], depth + 1)

        return Node(feature, threshold, left=left_subtree, right=right_subtree, value = info_gain)
        ### YOUR CODE HERE
        ##stopping criteria
        ##CAN USE common_thing()
        ##TO DO
        ##get_best_split()
        ##
        ##split()

        pass 

    def get_best_split(self, X, y, feat_idxs):
        best_split = None
        best_info_gain = -1  # Initialize with a negative value

        for feature in feat_idxs:
            feature_values = X[:, feature]
            unique_values = np.unique(feature_values)

            for threshold in unique_values:
                left_indices , right_indices = self.split(feature_values,threshold)

                if len(left_indices) == 0 or len(right_indices) == 0:
                    # Skip splits with no data on one side
                    continue

                info_gain = self.information_gain(y, feature_values, threshold)

                if info_gain > best_info_gain:
                    best_info_gain = info_gain
                    best_split = (feature, threshold, left_indices, right_indices, info_gain)

        return best_split
        ### YOUR CODE HERE
        ##
        ##information_gain()

        pass

    def information_gain(self, y, X_column, split_thresh):
        parent_entropy = entropy(y)

        # Split the labels into left and right based on the threshold
        left_indices, right_indices = self.split(X_column,split_thresh)

        # Calculate the weighted average of child entropies
        left_entropy = entropy(y[left_indices])
        right_entropy  = entropy(y[right_indices])

        # Calculate information gain
        info_gain = parent_entropy - (len(left_indices) / len(y)) * left_entropy - (len(right_indices) / len(y)) * right_entropy

        return info_gain
        ### YOUR CODE HERE

        # parent loss
        ##entropy()
        ##
        # generate split
        ##spit()
        ##
        # compute the weighted avg. of the loss for the children

        # information gain is difference in loss before vs. after split

        pass

    def split(self, X_column, split_thresh):
        left_indices = X_column <= split_thresh
        right_indices = X_column > split_thresh
        return left_indices, right_indices
        ### YOUR CODE HERE

        pass

    def traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value

        if x[node.feature] <= node.threshold:
            return self.traverse_tree(x, node.left)
        else:
            return self.traverse_tree(x, node.right)
        ### YOUR CODE HERE

        pass

    def common_thing(self, y):
        count = Counter(y)
        common_thing = count.most_common(1)[0][0]
        return common_thing


if __name__ == "__main__":

    data = datasets.load_iris()
    X, y = data.data, data.target

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    accuracy_depths = []
    for depth in range(1, 6):   
        clf = TreeRegressor(max_depth=depth)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        acc = accuracy(y_test, y_pred)
        accuracy_depths.append(acc)
        print("Accuracy at depth %d: %f" % (depth, acc))

    plt.figure()
    plt.plot(accuracy_depths)
    plt.xlabel("Depth")
    plt.ylabel("Accuracy")
    plt.show()

Accuracy at depth 1: 0.000000
Accuracy at depth 2: 0.000000


IndexError: list index out of range